# 00 — Markov chain, and where it breaks
Prologue, before Lecture 1. Lesson 0 in [PLAN](../knowledge/PLAN.md).

**Why this exists.** Your question is "why do we need neural nets at all?" The honest answer is "because the simpler thing stops working", and you'll see that best by building the simpler thing and watching it break.

**The simpler thing:** a Markov chain over letters. To guess the next letter, look at the last few letters, look up how often each letter followed them in real names, and pick one in proportion to those counts. No learning and no math, just a lookup table.

**Done when** you can explain, with numbers you measured here, why a lookup table can't use much context, and say in one sentence what a neural net would have to do differently.

## 1. Load the data
Scaffolding, already written. 32,000 names, one per line.

In [40]:
words = open("names.txt").read().splitlines()
# len(words), words[:8]

## 2. Count: one letter of context
For every name, walk through it pair by pair and count how often each letter follows each other letter. Add a marker such as `.` for "start" and "end", so the model also learns how names begin and end (`.emma.` → `.e`, `em`, `mm`, `ma`, `a.`).

Plain Python only: a dict of dicts or `collections.Counter`. Then look at it: what usually follows `q`? What usually ends a name?

In [41]:
from collections import Counter

# a list of all character pairs in the text,
# including start-of-word and end-of-word tokens
char_pairs: list[tuple[str, str]] = []

for word in words:
    char_pairs.append(("<s>", word[0]))  # add start-of-word token
    for i in range(len(word) - 1):
        char_pairs.append((word[i], word[i + 1]))

    char_pairs.append((word[-1], "</s>"))  # add end-of-word token


# a bigram table is a dictionary that maps a tuple of two chars
# to the number of times that bigram occurs in the text
bigram_table: dict[tuple[str, str], int] = Counter(char_pairs)

# bigram_table.most_common(10)  # show the 10 most common bigrams
# [
#     (bigram, count)
#     for bigram, count in bigram_table.items()
#     if bigram[1] == "</s>" 
# ]  # show bigrams that end with end-of-word token and their counts

## 3. Generate names
Start at `.`, pick the next letter in proportion to its counts (look at `random.choices` and its `weights=` argument), and repeat until you pick `.` again. Generate 10 names.

Before running: how name-like do you expect them to be?

In [42]:
from random import choices

NUM_NAMES_TO_GENERATE = 10

# dict[tuple[str, str], int] -> dict[str, list[tuple[str, int]]]
bigram_dict: dict[str, list[tuple[str, int]]] = {}
for (char1, char2), count in bigram_table.items():
    if char1 not in bigram_dict:
        bigram_dict[char1] = []
    bigram_dict[char1].append((char2, count))

# list(bigram_dict.items())[0]  # show the first item in the bigram_dict

def generate_name() -> str:
    name = ""
    current_char = "<s>"

    while True:
        next_chars, weights = zip(*bigram_dict[current_char])
        current_char = choices(next_chars, weights=weights)[0]

        if current_char == "</s>":
            break

        name += current_char

    return name.capitalize()

for _ in range(NUM_NAMES_TO_GENERATE):
    print(generate_name())

Ay
Zynican
Ridedigraneiaroleyvighan
Vanovillia
Avi
Brhakakancrie
Al
San
Ca
L


## 4. More context
Make the context length `n` a parameter (1, 2, 3, 4) and redo sections 2-3. For each `n`, measure:
- how many contexts are **possible** (27 symbols, `n` slots)
- how many you **actually saw**
- how many you saw only **once**

Predict before you run: do the names get better? What happens to those three numbers?

In [75]:
from collections import Counter
from random import choices

N = 4  # length of the character n-grams
NUM_NAMES_TO_GENERATE = 1_000

# a list of all character pairs in the text,
# including start-of-word and end-of-word tokens
char_pairs: list[tuple[str, ...]] = []

for word in words:
    # "<s>" + word + "</s>"
    # <s> + emma + </s> -> "<s>emma</s>"
    # ("<s>", "e"), ("e", "m"), ("m", "m"), ("m", "a"), ("a", "</s>") for N = 2
    # ("<s>", "e", "m"), ("e", "m", "m"), ("m", "m", "a"), ("m", "a", "</s>") for N = 3
    # "<s>kenny</s>"
    # ("<s>", "k"), ("k", "e"), ("e", "n"), ("n", "n"), ("n", "y"), ("y", "</s>") for N = 2
    word = "." + word + "."  # add start-of-word and end-of-word tokens

    for i in range(len(word) - (N - 1)):
        char_pairs.append(tuple(word[i : i + N]))  # add N-gram

# an n-gram table is a dictionary that maps a tuple of n chars
# to the number of times that n-gram occurs in the text
ngram_table: Counter[tuple[str, ...]] = Counter(char_pairs)

# list(ngram_table.items())[0]

# # dict[tuple[str, ...], int] -> dict[tuple[str, ...], list[tuple[str, int]]]
ngram_dict: dict[tuple[str, ...], list[tuple[str, int]]] = {}
for context, count in ngram_table.items():
    key = context[:-1]  # all but the last character
    next_char = context[-1]  # the last character
    if key not in ngram_dict:
        ngram_dict[key] = []
    ngram_dict[key].append((next_char, count))

# print(len(ngram_dict), sum(len(v) for v in ngram_dict.values() if len(v) == 1))  # number of contexts, number of next-character counts

# list(ngram_dict.items())[0]  # show the first item in the ngram_dict

"""
N = 2 | Possible contexts: 27^1 = 27 | Actually seen: 27 | Seen-only once: 0 
N = 3 | Possible contexts: 27^2 = 729 | Actually seen: 601 | Seen-only once: 71
N = 4 | Possible contexts: 27^3 = 19683 | Actually seen: 5659 | Seen-only once: 1651
N = 5 | Possible contexts: 27^4 = 531441 | Actually seen: 21757 | Seen-only once: 11999
N = 6 | Possible contexts: 27^5 = 14348907 | Actually seen: 38062 | Seen-only once: 27770
"""

def generate_sequence() -> str:
    if N < 2:
        raise ValueError("This function expects N >= 2.")

    starting_contexts = [
            context for context in ngram_dict
            if context[0] == "."
        ]
    if not starting_contexts:
            raise ValueError("No start contexts found")

    starting_weights = [
            sum(count for _, count in ngram_dict[context])
            for context in starting_contexts
        ]
    current_context = choices(
            starting_contexts,
            weights=starting_weights,
        )[0]
    name = "".join(current_context[1:])

    while True:
        options = ngram_dict.get(current_context)
        if not options:
            raise ValueError(
                f"No next-character counts for context {current_context!r}"
            )

        next_chars, weights = zip(*options)
        next_char = choices(next_chars, weights=weights)[0]

        if next_char == ".":
            break

        name += next_char
        current_context = (*current_context[1:], next_char)

    return name

# for _ in range(NUM_NAMES_TO_GENERATE):
#     print(generate_sequence())

## 5. Break it
Push `n` up to 5 or 6, then generate.
- Are the names new, or copies of real ones? Check with `name in words`.
- Ask for the next-letter counts after a context that never appears in the data, like `'zzq'`. What can the table say?

Predict first, then run.

In [76]:
copied_names = 0
for _ in range(NUM_NAMES_TO_GENERATE):
    try:
        generated_name = generate_sequence()
        if generated_name in words:
            copied_names += 1
    except ValueError as e:
        print(f"Error generating name: {e}")

print(f"Number of copied names: {copied_names}/{NUM_NAMES_TO_GENERATE}")

"""
N = 2 | Number of copied names: 94/1000
N = 3 | Number of copied names: 209/1000
N = 4 | Number of copied names: 323/1000
N = 5 | Number of copied names: 574/1000
N = 6 | Number of copied names: 850/1000
"""

if N == 4:
    try:
        ngram_dict[("z", "z", "q")]  # this should raise a KeyError
    except KeyError:
        print("KeyError raised as expected for context ('z', 'z', 'q')")

Number of copied names: 316/1000
KeyError raised as expected for context ('z', 'z', 'q')


## 6. In your own words
Answer these in the markdown cell below (no code):
1. Why does more context help in theory but hurt in practice for a table?
2. The table treats `'ann'` and `'enn'` as completely unrelated. Why is that wasteful?
3. Suppose a model had a **fixed** number of adjustable knobs, shared across all contexts, so that similar contexts got similar guesses. What problem would that fix?

(Q3 describes a neural net. Lecture 3 opens with exactly this problem: search "exponentially with the length of the context", ~1% in.)

**1. Why does more context help in theory but hurt in practice?**

*Helps in theory:* more context narrows the guess. After `u` almost any letter can come next. After `q` it's `u` about 9 times in 10. After `ma` at the start of a name it's usually `r` (Mar...), then `l` or `k`, almost never `q`. The more letters you know, the fewer next letters make sense.

*Hurts in practice:* the table needs a row for every possible context, and every extra letter multiplies the rows by 27. The data can't keep up:

| N | possible | actually seen | seen % | forced (one next letter) | copies |
|---|---|---|---|---|---|
| 2 | 27 | 27 | 100% | 0% | 9% |
| 4 | 19,683 | 5,659 | 29% | 29% | 32% |
| 6 | 14,348,907 | 38,062 | 0.27% | 73% | 85% |

There are only 228,146 next-letter examples in the whole dataset, so the table can never be more than a sliver full. The rows it does fill have one or two examples each. A row with one example has only one possible next letter, so it's **forced**: once the generator steps into it, it replays the one name it saw. That's why the names looked "meaningful" at N = 6: 85% of them were real names copied from the data. And a context it never saw (`zzq`) has no row at all, so the table has no opinion, not even "unlikely".

**2. Why is treating `ann` and `enn` as unrelated wasteful?**

They behave alike. At N = 4, `ann` (825 examples), `enn` (240) and `onn` (194) all say "next is usually `a`, `e`, `i`, end, or `y`". But to the table they are separate keys, as unrelated as `ann` and `xqz`. So when a context like `unn` has only 15 examples, the table bases its whole guess on those 15 and ignores the 1,200+ examples from its siblings. Evidence about one context can't help a similar one, so every row starves on its own data.

Not every lookalike is similar, though: `ynn` (Lynn, Flynn) usually *ends* the name. So the fix can't be a hand-written rule like "same last two letters = same". The model has to *learn* which contexts behave alike.

**3. What would a fixed set of shared knobs fix?**

Two things:
- **Sharing:** if `ann`, `enn`, `onn` and `unn` are handled by the same knobs instead of separate count rows, then `unn` borrows from what was learned on `ann`. Rare and even never-seen contexts get a sensible guess instead of nothing.
- **No explosion:** the number of knobs is fixed. Going from 3 to 8 letters of context doesn't multiply the model's size by 27⁵. It doesn't need a separate row for 14 million contexts that the data will never fill.

That's a neural net. Lecture 3 opens with exactly this problem (search "exponentially with the length of the context", ~1% in).
